In [0]:
# documentação:
# objetivo: uma tabela fato histórica que rastreie o ciclo de vida completo de cada bid desde a abertura até finalização (ou até hoje se ainda em andamento).
# 
# justificativa: 
# - permitir análises temporais do comportamento dos bids ao longo do tempo
# - identificar quantos bids estão abertos, em andamento ou encerrados em cada mês
# - facilitar cálculos de métricas como taxa de conversão, tempo médio de fechamento, pipeline de vendas mensal
# 
# metodologia:
# 1. geração de calendário mensal: para cada bid, criar registros para todos os meses desde a abertura até a finalização (ou mês atual)
# 2. classificação de status mensal:
#    - flg_aberto = 1: apenas no mês de abertura do bid
#    - flg_encerrado = 1: apenas no mês de finalização do bid
#    - flg_andamento = 1: todos os meses entre abertura e finalização (exclusive)
# 3. enriquecimento: adicionar atributos descritivos do bid (cliente, situação, motivo perda, concorrência)
# 4. timestamp de processamento: registrar momento da carga com ajuste de fuso horário (-3h)
# 
# premissas:
# - bids sem data de finalização são considerados em andamento até o mês atual (ou mês de abertura se aberto no futuro)
# - um bid pode ter apenas um registro por mês
# - id_situacao_bid 571 representam situações "em andamento"
# - bids com dt_finalizacao anterior à dt_abertura são filtrados (dados inconsistentes)
# 
# observações:
# - utiliza explode com sequence para gerar a série temporal mensal
# - date_format converte datas para formato yyyymm (integer) para join com dimensão tempo
# - coalesce garante valores default para campos opcionais
# - greatest garante que bids abertos no futuro não gerem sequências inválidas


df_fat_bid_historico = spark.sql("""
with cte_bid_com_periodo as (
  select
  a.id_bid
  ,a.dt_abertura_bid
  ,a.dt_cadastro
  ,a.in_bid
  ,a.dt_finalizacao_bid
  ,a.in_ganhou
  ,a.ds_bid_motivo_perda
  ,a.ds_concorrencia
  ,a.id_contrato
--   ,coalesce(a.in_ganhou, 'Não informado') as in_ganhou
--   ,coalesce(a.ds_bid_motivo_perda, 'Não informado') as ds_bid_motivo_perda
--   ,coalesce(a.ds_concorrencia, 'Não informado') as ds_concorrencia
      -- definir data final: se finalizado usa dt_finalizacao, senão usa o maior entre abertura e hoje
  ,case 
    when a.dt_finalizacao_bid is not null then a.dt_finalizacao_bid
    else greatest(a.dt_abertura_bid, last_day(current_date()))
  end as dt_fim_periodo
  from silver.bid.entidade_bid as a
  where a.dt_abertura_bid is not null
      -- garantir que a data de finalização não seja anterior à abertura
      and (a.dt_finalizacao_bid is null or a.dt_finalizacao_bid >= a.dt_abertura_bid)
),
cte_meses_explodidos as (
  select
    a.id_bid
    ,a.id_contrato
    ,a.dt_abertura_bid
    ,a.dt_finalizacao_bid
    ,a.in_ganhou
    ,a.ds_bid_motivo_perda
    ,a.ds_concorrencia
    -- gerar sequência de meses entre abertura e fim do período
    ,explode(
        sequence(
            trunc(a.dt_abertura_bid, 'month'),
            trunc(a.dt_fim_periodo, 'month'),
            interval 1 month
        )
    ) as dt_mes_referencia
    from cte_bid_com_periodo as a
),
cte_classificacao_status as (
  select
    a.id_bid
    ,a.id_contrato
    ,a.dt_abertura_bid
    ,a.dt_finalizacao_bid
    ,a.in_ganhou
    ,a.ds_bid_motivo_perda
    ,a.ds_concorrencia
    ,a.dt_mes_referencia
    -- flags de status do bid no mês
    ,case 
        when trunc(a.dt_abertura_bid, 'month') = a.dt_mes_referencia then 1 
        else 0 
    end as flg_aberto
    ,case 
        when a.dt_finalizacao_bid is not null 
            and trunc(a.dt_finalizacao_bid, 'month') = a.dt_mes_referencia then 1
        else 0
    end as flg_encerrado
    ,case
        when trunc(a.dt_abertura_bid, 'month') < a.dt_mes_referencia
            and (
                a.dt_finalizacao_bid is null 
                or trunc(a.dt_finalizacao_bid, 'month') > a.dt_mes_referencia
            ) then 1
        else 0
    end as flg_andamento
  from cte_meses_explodidos as a
)
select
    a.id_bid
    ,a.id_contrato
    ,a.dt_abertura_bid
    ,case 
        when a.flg_encerrado = 0 
        then null 
        else a.dt_finalizacao_bid 
    end as dt_finalizacao_bid
    ,case 
        when a.flg_encerrado = 0 
        then null 
        else a.in_ganhou 
    end as in_ganhou
    ,case 
        when a.flg_encerrado = 0 
        then null 
        else a.ds_bid_motivo_perda 
    end as ds_bid_motivo_perda
    ,case 
        when a.flg_encerrado = 0 
        then null 
        else a.ds_concorrencia 
    end as ds_concorrencia
    ,cast(date_format(a.dt_mes_referencia, 'yyyyMM') as int) as nr_ano_mes
    ,a.dt_mes_referencia as dt_referencia
    ,a.flg_aberto
    ,a.flg_encerrado
    ,a.flg_andamento
    ,current_timestamp() - interval 3 hours as dt_processamento
from cte_classificacao_status as a
order by a.id_bid, a.dt_mes_referencia
""")

# df_fat_bid_historico.createOrReplaceTempView('vw_fat_bid_historico')

In [0]:
df_fat_bid_historico\
    .write\
    .mode("overwrite")\
    .option("mergeSchema", True)\
    .format("delta")\
    .saveAsTable("gold.bid.fato_bid_historico")